# Evaluating a fixed policy — the time-to-go of a hand-written control law

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro860/double_integrator_policy_evaluation.ipynb)

Value iteration answers *what is the best policy?* This notebook asks the simpler question that comes first: **given** a policy $\pi$, what is its cost-to-go $J^\pi(x)$? The answer is **policy evaluation**: the same backward sweep as value iteration, with the minimization over $u$ removed because the action is fixed by the law,
$$J^\pi(x) \leftarrow g\big(x, \pi(x)\big)\,\Delta t + J^\pi\Big(x + f\big(x, \pi(x)\big)\,\Delta t\Big) .$$

The plant is a unit mass on a line, force $|u| \le 1$, and the cost counts the time spent outside a small zone around the origin, so $J^\pi$ is the **time the policy takes to bring the mass home**. The notebook shows how to

1. write a policy as a block that closes the loop,
2. evaluate its time-to-go on the grid,
3. compare it with the optimal time-to-go from value iteration, and
4. check $J^\pi(x_0)$ against the time a simulated run actually takes.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import (
    Controller,
    DoubleIntegrator,
    DynamicProgrammingPlanner,
    PlanningProblem,
    PolicyEvaluator,
    StateSpaceGrid,
    TimeCost,
)

## 1. Plant, cost and grid

Same problem as the value-iteration notebook. The grid is built once and shared by the evaluator and, later, by the optimal planner, so the two cost-to-go fields live on the same nodes.

In [ ]:
EPS = 0.1  # radius of the target zone around x = 0 where the cost stops
INF = 20.0  # price of leaving the box, and cap for the plots
X_GRID = (201, 201)
U_GRID = (3,)
DT = 0.05
X0 = np.array([0.5, 0.0])

plant = DoubleIntegrator()
plant.state.lower_bound[:] = [-2.0, -2.0]
plant.state.upper_bound[:] = [+2.0, +2.0]
plant.inputs["u"].lower_bound = np.array([-1.0])
plant.inputs["u"].upper_bound = np.array([+1.0])
plant.x0 = X0

cost = TimeCost.from_system(plant, eps=EPS)
problem = PlanningProblem(plant, x_goal=np.zeros(2), cost=cost)
grid = StateSpaceGrid(problem, x_grid_shape=X_GRID, u_grid_shape=U_GRID, dt=DT)

## 2. A hand-written policy

A policy is a block whose output is the control law. This one is a saturated proportional-derivative law on the position $p$ and the speed $v$:
$$u = \pi(x) = \operatorname{clip}\big(-k_p\, p - k_d\, v,\; -1,\; 1\big).$$
Declaring `feedback_profile = "state"` tells the tools that the block reads the full state, so `policy @ plant` closes the loop and `plot_control_law` draws the map $u = \pi(p, v)$.

In [ ]:
class PositioningPolicy(Controller):
    """u = clip(-kp p - kd v, -1, 1): a saturated proportional-derivative law."""

    feedback_profile = "state"  # the block reads the plant state x = [p, v]

    def __init__(self, kp=0.5, kd=0.5):
        super().__init__()
        self.name = "Positioning policy"
        self.params = {"kp": kp, "kd": kd}
        self.add_input_port("x", dim=2)
        self.add_output_port("u", dim=1, function=self.ctl, dependencies="all")

    def ctl(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        kp, kd = params["kp"], params["kd"]
        p, v = u  # the block's input is the plant state

        force = -kp * p - kd * v
        return np.array([np.clip(force, -1.0, 1.0)])


policy = PositioningPolicy()
policy.plot_control_law(bounds=((-2.0, 2.0), (-2.0, 2.0)), vmin=-1.0, vmax=1.0)

## 3. Policy evaluation

`PolicyEvaluator` runs the fixed-policy sweep on the grid: at every node it applies $\pi$, steps the dynamics once, and adds the interpolated cost-to-go of the arrival state, until $J^\pi$ stops changing. The options (no discount, tolerance, price of leaving the box) are those of a `DynamicProgrammingPlanner` on the same grid — the planner is kept for the comparison of the next section.

In [ ]:
planner = DynamicProgrammingPlanner(
    problem, grid=grid, alpha=1.0, tol=1e-3, max_iterations=2000, out_of_bound_cost=INF
)
evaluator = PolicyEvaluator(problem, grid=grid, policy=policy, options=planner.options)
J_pi = evaluator.solve()

print(f"J_pi(x0) = {evaluator.value_at(X0):.2f} s")
evaluator.plot_cost2go(vmax=INF, show_3d=True, title="Time-to-go of the PD policy")
evaluator.plot_cost2go(vmax=INF, title="Time-to-go of the PD policy")

## 4. Comparison with the optimal policy

Value iteration on the same grid gives $J^*$. No policy can beat it, so $J^\pi(x) - J^*(x) \ge 0$ everywhere: the map of that difference shows *where* the hand-written law loses the most time. Interpolation on the grid can make the difference slightly negative in places; that is discretization error, not a better policy.

In [ ]:
planner.solve()
J_star = planner.result.J

print(f"J*(x0)   = {planner.value_at(X0):.2f} s")
print(f"J_pi(x0) = {evaluator.value_at(X0):.2f} s")
planner.plot_cost2go(jmax=INF, title="Optimal time-to-go $J^*$")
grid.plot_value(J_pi - J_star, vmin=0.0, vmax=5.0, title="Time lost by the PD policy, $J^\\pi - J^*$")

## 5. Simulated runs against the evaluated cost-to-go

`policy @ plant` closes the loop; it is simulated with the same Euler step as the grid, and the trajectory is drawn over the optimal policy map.

In [ ]:
plant.x0 = X0
loop = policy @ plant

traj = loop.compute_trajectory(tf=15.0, n_steps=int(15.0 / DT) + 1, solver="euler")
loop.plot_trajectory(traj)
planner.plot_policy(trajectory=traj)

$J^\pi(x_0)$ predicts the time the closed loop needs to enter the target zone. The table below checks it from several starts: the arrival time read off each run, and the running cost integrated along it — the time spent outside the zone, the quantity the evaluator sweeps for; the two agree because the mass stays in the zone once it gets there. The evaluated value sits about a second **above** both: the sweep interpolates $J^\pi$ linearly between nodes at every step, and the slow spiral of this law crosses many nodes, so the interpolation bias accumulates. A finer grid and a smaller step close the gap (section 6).

In [ ]:
def arrival_time(traj):
    """First time the state enters the target zone (inf if it never does)."""
    inside = np.linalg.norm(traj.x, axis=0) < EPS
    return traj.t[inside][0] if inside.any() else np.inf


print("x0              J_pi(x0)   arrival   time outside the zone")
for x0 in ([0.5, 0.0], [1.5, 0.0], [-1.0, 1.0], [0.0, -1.5]):
    plant.x0 = np.array(x0)
    traj = loop.compute_trajectory(tf=15.0, n_steps=int(15.0 / DT) + 1, solver="euler", verbose=False)
    J_run = loop.compute_cost(cost, of=plant, traj=traj)
    print(f"{np.round(x0, 2)!s:<16}{evaluator.value_at(x0):8.2f}  {arrival_time(traj):8.2f}  {J_run:8.2f}")

## 6. Things to try

1. **Gains.** Change `kp` and `kd` (try `PositioningPolicy(kp=2.0, kd=2.0)`), re-evaluate, and compare the time lost. Does a stiffer law always arrive sooner under a force limit?
2. **A better policy.** Write a second `Controller` whose law switches the force between $-1$ and $+1$ according to the sign of a function of $(p, v)$ — the shape of the optimal policy map in section 4 is the hint — evaluate it, and check that its time-to-go approaches $J^*$.
3. **Prediction versus simulation.** Rebuild the grid with `X_GRID = (401, 401)` and `DT = 0.025`, re-evaluate the PD law, and compare $J^\pi(x_0)$ with the simulated arrival time again (the gap should fall from about a second to a tenth). Then do the same for your new policy: when the two disagree, decide whether the grid, the time step, or the policy is responsible.